# Genomic–MRI fusion results
Comparison of **late fusion** and **cross-attention** using saved GBM/LGG experiment summaries. No training or raw MRI loading is performed. Run the following cells in order. Saved average rows are excluded from fold statistics. The MRI encoder may differ between historical runs; these tables summarize the saved experiments, not necessarily a controlled comparison of identical settings.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "main.py").is_file() and (p / "result").is_dir())
RESULT_DIR = ROOT / "result"
OUTPUT_DIR = RESULT_DIR / "comparisons/genomic_mri"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS = {"Late fusion": "multimodal_mri_late_fusion",
          "Cross-attention": "multimodal_mri_cross_attention"}
COHORTS = {"GBM": ["normalized_zscore_gbm.csv", "gbm", "gbm.csv"],
           "LGG": ["normalized_zscore_lgg.csv", "lgg", "lgg.csv"]}
METRICS = {"train_cindex": "Training C-index", "final_val_cindex": "Validation C-index",
           "train_loss": "Training loss", "final_val_loss": "Validation loss"}

rows, fold_frames = [], []
for cohort, datasets in COHORTS.items():
    for model_name, model_dir in MODELS.items():
        summary_path = next((RESULT_DIR / model_dir / dataset / "summary.csv"
                             for dataset in datasets
                             if (RESULT_DIR / model_dir / dataset / "summary.csv").exists()), None)
        row = {"Cohort": cohort, "Model": model_name, "Status": "No saved results"}
        if summary_path is None:
            rows.append(row)
            continue
        frame = pd.read_csv(summary_path)
        folds = pd.to_numeric(frame["fold"], errors="coerce")
        frame = frame.loc[folds.notna() & (folds >= 0) & (folds % 1 == 0)].copy()
        frame["fold"] = pd.to_numeric(frame["fold"]).astype(int)
        if frame.empty:
            row["Status"] = "No fold rows"
            rows.append(row)
            continue
        if frame["fold"].duplicated().any():
            raise ValueError(f"Duplicate fold rows in {summary_path}")
        row.update({"Status": "Available", "Folds": len(frame),
                    "Dataset": summary_path.parent.name, "Summary": str(summary_path.relative_to(ROOT))})
        loss_names = frame["loss_func"].dropna().astype(str).unique() if "loss_func" in frame else []
        row["Loss function"] = ", ".join(loss_names) if len(loss_names) else "Unknown (not recorded)"
        for metric in METRICS:
            frame[metric] = pd.to_numeric(frame[metric], errors="coerce").replace([np.inf, -np.inf], np.nan)
            row[f"{metric}_mean"] = frame[metric].mean()
            row[f"{metric}_std"] = frame[metric].std()
            row[f"{metric}_valid_folds"] = frame[metric].count()
        row["Average C-index"] = (row["train_cindex_mean"] + row["final_val_cindex_mean"]) / 2
        row["C-index gap"] = (frame["train_cindex"] - frame["final_val_cindex"]).mean()
        rows.append(row)
        fold_frames.append(frame.assign(Cohort=cohort, Model=model_name,
                                         Dataset=summary_path.parent.name, **{"Loss function": row["Loss function"]}))

mean_results = pd.DataFrame(rows)
for column in ["Folds", "Loss function", "Average C-index"]:
    if column not in mean_results:
        mean_results[column] = np.nan
mean_results["Folds"] = mean_results["Folds"].astype("Int64")
fold_results = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()


## Mean training and validation results
Values are mean ± standard deviation across available folds. **Average C-index** is the average of the training and validation means, not an average row across models. Missing experiments remain marked as missing. An unknown loss name is not inferred from current script settings. Loss magnitudes require matching loss objectives and survival intervals.

In [ ]:
comparison_table = mean_results[["Cohort", "Model", "Status", "Folds", "Loss function"]].copy()
for metric, label in METRICS.items():
    comparison_table[label] = [
        "—" if pd.isna(row.get(f"{metric}_mean")) else f"{row[f'{metric}_mean']:.3f}"
        if pd.isna(row.get(f"{metric}_std"))
        else f"{row[f'{metric}_mean']:.3f} ± {row[f'{metric}_std']:.3f}"
        for _, row in mean_results.iterrows()
    ]
comparison_table["Average C-index"] = mean_results["Average C-index"].round(3)
display(comparison_table)
mean_results.to_csv(OUTPUT_DIR / "genomic_mri_fusion_mean_results.csv", index=False)
comparison_table.to_csv(OUTPUT_DIR / "genomic_mri_fusion_comparison_table.csv", index=False)
print("Average C-index = (mean training C-index + mean validation C-index) / 2.")
print("This average is descriptive; validation C-index is the held-out ranking metric.")
print("Saved tables to:", OUTPUT_DIR)


Cohort,Model,Status,Folds,Loss function,Training C-index,Validation C-index,Training loss,Validation loss,Average C-index
GBM,Late fusion,Available,5,NLLSurvLoss,0.925 ± 0.020,0.375 ± 0.195,0.121 ± 0.054,3.996 ± 0.908,0.650
GBM,Cross-attention,Available,5,NLLSurvLoss,0.913 ± 0.007,0.425 ± 0.171,0.149 ± 0.083,5.025 ± 2.047,0.669
LGG,Late fusion,Available,5,NLLSurvLoss,0.965 ± 0.006,0.513 ± 0.345,0.070 ± 0.027,1.646 ± 0.455,0.739
LGG,Cross-attention,No saved results,<NA>,NaN,—,—,—,—,NaN


Average C-index = (mean training C-index + mean validation C-index) / 2.
This average is descriptive; validation C-index is the held-out ranking metric.
Saved tables to: /home/licongcong/Desktop/learning/medical_image_classification/BrainCanerSurvival/result/comparisons/genomic_mri


## Every fold
Training and held-out validation results for every saved numeric fold. Fold counts can differ between experiments.

In [ ]:
if fold_results.empty:
    print("No saved fold results are available.")
else:
    fold_columns = ["Cohort", "Model", "Dataset", "Loss function", "fold", *METRICS]
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(fold_results.sort_values(["Cohort", "Model", "fold"])[fold_columns].round(4))
    fold_results.to_csv(OUTPUT_DIR / "genomic_mri_fusion_fold_results.csv", index=False)


Cohort,Model,Dataset,Loss function,fold,train_cindex,final_val_cindex,train_loss,final_val_loss
GBM,Cross-attention,normalized_zscore_gbm.csv,NLLSurvLoss,0,0.9115,0.5625,0.2632,6.4228
GBM,Cross-attention,normalized_zscore_gbm.csv,NLLSurvLoss,1,0.9229,0.3529,0.1907,4.5695
GBM,Cross-attention,normalized_zscore_gbm.csv,NLLSurvLoss,2,0.9053,0.3125,0.0592,6.6773
GBM,Cross-attention,normalized_zscore_gbm.csv,NLLSurvLoss,3,0.9070,0.6471,0.1547,1.6658
GBM,Cross-attention,normalized_zscore_gbm.csv,NLLSurvLoss,4,0.9172,0.2500,0.0792,5.7911
GBM,Late fusion,normalized_zscore_gbm.csv,NLLSurvLoss,0,0.8928,0.6758,0.1866,3.9675
GBM,Late fusion,normalized_zscore_gbm.csv,NLLSurvLoss,1,0.9387,0.2353,0.1043,5.0001
GBM,Late fusion,normalized_zscore_gbm.csv,NLLSurvLoss,2,0.9199,0.4570,0.1693,4.7536
GBM,Late fusion,normalized_zscore_gbm.csv,NLLSurvLoss,3,0.9427,0.3015,0.0679,2.8061
GBM,Late fusion,normalized_zscore_gbm.csv,NLLSurvLoss,4,0.9303,0.2031,0.0760,3.4506
